# Scale to zero - step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will check database compute state, stop it, and restart it, dropping cost to 0/hr when idle.

Each step uses the real interface: SDK for control-plane operations, SQL for queries.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set.

## Setup - initialize

In [ ]:
import os, sys, timesys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))from lakebase import client, _envfrom databricks.sdk.service.postgres import DatabaseInstancePROJECT = os.environ['LB_PROJECT']w = client()def get_instance_state(w, project):    inst = w.postgres.get_database_instance(name=f"projects/{project}")    ep = inst.endpoints[0] if inst.endpoints else None    return {        "endpoint_id": ep.endpoint_id if ep else "?",        "state": ep.status.current_state.value if (ep and ep.status and ep.status.current_state) else "UNKNOWN",        "nodes": inst.nodes if inst.nodes else 0,        "effective_stopped": inst.effective_stopped if inst.effective_stopped else False,        "cu": inst.compute_unit if inst.compute_unit else "?",    }print('connected as', w.current_user.me().user_name)print(f'project={PROJECT}')

## Step 1 - Check the current compute state

Query the project's compute endpoint to see its current state. This step uses the SDK, not SQL.

In [ ]:
state = get_instance_state(w, PROJECT)print(f"  endpoint_id: {state['endpoint_id']}")print(f"  state: {state['state']}")print(f"  nodes: {state['nodes']}")print(f"  cu: {state['cu']}")if state['effective_stopped']:    print(f"  effective_stopped: {state['effective_stopped']}")

## Step 2 - Stop the compute

Call the Lakebase API to stop compute. The script polls until stopped and prints timing.

In [ ]:
t0 = time.time()w.postgres.update_database_instance(    name=f"projects/{PROJECT}",    database_instance=DatabaseInstance(stopped=True))# Poll until stoppedwhile time.time() - t0 < 30:    state = get_instance_state(w, PROJECT)    if state["state"] == "STOPPED" and state["nodes"] == 0 and state["effective_stopped"]:        elapsed = time.time() - t0        print(f"  stopped in ~{elapsed:.1f}s; nodes: {state['nodes']}; effective_stopped: true")        break    time.sleep(0.5)

## Step 3 - Verify the stop

Check the state again to confirm compute is stopped.

In [ ]:
state = get_instance_state(w, PROJECT)print(f"  endpoint_id: {state['endpoint_id']}")print(f"  state: {state['state']}")print(f"  nodes: {state['nodes']}")print(f"  effective_stopped: {state['effective_stopped']}")

## Step 4 - Restart the compute

Call the Lakebase API to start compute. The script polls until active and prints timing.

In [ ]:
t0 = time.time()w.postgres.update_database_instance(    name=f"projects/{PROJECT}",    database_instance=DatabaseInstance(stopped=False))# Poll until activewhile time.time() - t0 < 60:    state = get_instance_state(w, PROJECT)    if state["state"] == "ACTIVE" and state["nodes"] > 0:        elapsed = time.time() - t0        print(f"  started in ~{elapsed:.1f}s; state: {state['state']}; nodes: {state['nodes']}")        break    time.sleep(0.5)

## What you learned

You controlled database compute costs with scale-to-zero: stopping compute drops cost to 0/hr, restarting takes about 7 seconds, and data remains untouched. No long-running idle charges.

**Next:** Write to analytics - combine Postgres operations with Databricks lakehouse via foreign tables.